# CS-4063 — Natural Language Processing, Assignment 3
## Transformer-based Review Understanding with RAG Enhanced Explanation Generation

**Student ID:** i232545  
**Course:** CS-4063 Natural Language Processing  
**Due Date:** 29-04-2026

---

## Execution Instructions

**Prerequisites:**
```
pip install torch numpy pandas matplotlib scikit-learn tqdm
```

**Dataset Setup:**
- All `.json.gz` files must be inside the `Dataset/` folder (sibling to this notebook)
- Required files: `sports.json.gz`, `beauty.json.gz`, `cellphones.json.gz`

**Run all cells top-to-bottom.** Directories `models/` and `results/` are created automatically.

---

## Chunk 1: Project Setup + Dataset Loading + Preprocessing Pipeline

**This commit covers:**
- Project directory setup
- Dataset loading from `.json.gz` files
- Category sampling (10k–15k per category, 3 categories)
- Full preprocessing pipeline (cleaning → tokenization → vocabulary → numericalization → padding/truncation)
- Train / Validation / Test split (70 / 15 / 15)
- Saving preprocessed data and vocabulary to disk

---
## Section 0: Imports & Global Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import json
import gzip
import random
import pickle
import string
from collections import Counter
from pathlib import Path

# ── Third-party ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports successful.")

In [ ]:
# ── Project-wide constants ─────────────────────────────────────────────────────

# Paths (relative to notebook location)
DATASET_DIR   = Path("Dataset")
MODELS_DIR    = Path("models")
RESULTS_DIR   = Path("results")
PLOTS_DIR     = RESULTS_DIR / "learning_curves"

# Dataset construction
CATEGORIES = {
    "sports":      DATASET_DIR / "sports.json.gz",
    "beauty":      DATASET_DIR / "beauty.json.gz",
    "cellphones":  DATASET_DIR / "cellphones.json.gz",
}
SAMPLES_PER_CATEGORY = 12000   # target ~12k per category → ~36k total

# Preprocessing
MAX_SEQ_LEN  = 128             # max tokens per review (truncate/pad to this)
MIN_FREQ     = 2               # minimum token frequency to enter vocabulary

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

# Sentiment label mapping
SENTIMENT_MAP = {
    1: 0,  # Negative
    2: 0,  # Negative
    3: 1,  # Neutral
    4: 2,  # Positive
    5: 2,  # Positive
}
SENTIMENT_LABELS = ["Negative", "Neutral", "Positive"]

# Create output directories
for d in [MODELS_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Dataset dir   : {DATASET_DIR.resolve()}")
print(f"  Models dir    : {MODELS_DIR.resolve()}")
print(f"  Results dir   : {RESULTS_DIR.resolve()}")
print(f"  Samples/cat   : {SAMPLES_PER_CATEGORY}")
print(f"  Max seq len   : {MAX_SEQ_LEN}")

---
## Section 1: Dataset Loading

The Amazon Reviews dataset is stored as `.json.gz` files where each line is a separate JSON object. We load each category file, extract the `reviewText` and `overall` (star rating) fields, drop rows with missing values, and sample the required number of reviews.

In [ ]:
def load_category(filepath: Path, n_samples: int, category_name: str) -> pd.DataFrame:
    """
    Load up to n_samples reviews from a .json.gz Amazon Reviews file.
    Only keeps rows with non-empty reviewText and a valid star rating (1-5).

    Args:
        filepath     : path to the .json.gz file
        n_samples    : maximum number of rows to sample
        category_name: label stored in the 'category' column

    Returns:
        DataFrame with columns [review_text, rating, sentiment, category]
    """
    records = []
    print(f"Loading '{category_name}' from {filepath} ...", flush=True)

    with gzip.open(filepath, "rt", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, desc=f"  Reading {category_name}", unit=" lines"):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            text   = obj.get("reviewText", "") or ""
            rating = obj.get("overall", None)

            # Skip if text is empty or rating is missing / out of range
            if not text.strip():
                continue
            if rating is None or int(rating) not in SENTIMENT_MAP:
                continue

            records.append({
                "review_text": text.strip(),
                "rating":      int(rating),
            })

            # Stop early once we have enough valid records
            if len(records) >= n_samples * 3:   # over-sample then downsample
                break

    df = pd.DataFrame(records)

    # Stratified downsample to n_samples to keep rating balance
    if len(df) > n_samples:
        df = df.groupby("rating", group_keys=False).apply(
            lambda g: g.sample(frac=n_samples / len(df), random_state=SEED)
        ).reset_index(drop=True)
        # Guarantee we don't exceed n_samples
        df = df.sample(n=min(n_samples, len(df)), random_state=SEED).reset_index(drop=True)

    df["sentiment"] = df["rating"].map(SENTIMENT_MAP)
    df["category"]  = category_name

    print(f"  → Loaded {len(df):,} reviews for '{category_name}'")
    return df

In [ ]:
# Load all three categories
dfs = []
for cat_name, cat_path in CATEGORIES.items():
    df_cat = load_category(cat_path, SAMPLES_PER_CATEGORY, cat_name)
    dfs.append(df_cat)

# Combine into one DataFrame
data = pd.concat(dfs, ignore_index=True)
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f"\nTotal dataset size: {len(data):,} reviews")
print(f"Columns: {list(data.columns)}")

In [ ]:
# ── Dataset statistics ─────────────────────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total samples  : {len(data):,}")
print()

print("Samples per category:")
print(data["category"].value_counts().to_string())
print()

print("Rating distribution:")
print(data["rating"].value_counts().sort_index().to_string())
print()

print("Sentiment distribution:")
sent_counts = data["sentiment"].value_counts().sort_index()
for idx, count in sent_counts.items():
    label = SENTIMENT_LABELS[idx]
    pct   = 100 * count / len(data)
    print(f"  {label:10s} ({idx}): {count:,}  ({pct:.1f}%)")

In [ ]:
# ── Visualize rating distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rating distribution
rating_counts = data["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values,
            color=["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#1f77b4"],
            edgecolor="white")
axes[0].set_title("Rating Distribution (1–5 stars)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(axes[0].patches, rating_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 100,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)

# Sentiment distribution
sent_counts = data["sentiment"].value_counts().sort_index()
colors = ["#d62728", "#bcbd22", "#2ca02c"]
axes[1].bar([SENTIMENT_LABELS[i] for i in sent_counts.index],
            sent_counts.values, color=colors, edgecolor="white")
axes[1].set_title("Sentiment Class Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Count")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(axes[1].patches, sent_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 100,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "dataset_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved.")

---
## Section 2: Derived Feature Definition

As required by Part A, we define a second task beyond sentiment classification. We choose **Review Length Category** as the derived feature.

**Motivation:** Review length correlates meaningfully with reviewer engagement. Very short reviews (few words) tend to be uninformative (e.g., "Great!"). Medium reviews often provide balanced feedback. Long reviews tend to be detailed and opinionated. Length is directly derivable from text alone — no external signals needed — making it a valid multi-task learning target. The model jointly learning to estimate length category should also encourage it to learn global structural properties of the review text.

**Label definition:**

| Class | Label | Word Count Range |
|---|---|---|
| 0 | Short  | 1–30 words   |
| 1 | Medium | 31–100 words |
| 2 | Long   | 101+ words   |

In [ ]:
# ── Derived feature: Review length category ────────────────────────────────────
LENGTH_LABELS = ["Short (1-30)", "Medium (31-100)", "Long (101+)"]

def assign_length_category(text: str) -> int:
    """
    Assign a 3-class length label based on word count.
      0 = Short  : 1–30 words
      1 = Medium : 31–100 words
      2 = Long   : 101+ words
    """
    n_words = len(text.split())
    if n_words <= 30:
        return 0
    elif n_words <= 100:
        return 1
    else:
        return 2

data["length_label"] = data["review_text"].apply(assign_length_category)

print("Derived feature (review length category) distribution:")
for idx, label in enumerate(LENGTH_LABELS):
    count = (data["length_label"] == idx).sum()
    pct   = 100 * count / len(data)
    print(f"  {label:20s}: {count:,}  ({pct:.1f}%)")

---
## Section 3: Train / Validation / Test Split

We split **before** building the vocabulary to prevent data leakage. The vocabulary is constructed exclusively from the training set in Section 4.

In [ ]:
# Stratified split by sentiment label to preserve class balance across splits
# Step 1: Train vs Rest (85%)
train_df, temp_df = train_test_split(
    data,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=data["sentiment"],
    random_state=SEED
)

# Step 2: Val vs Test (50/50 of the remaining 30%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["sentiment"],
    random_state=SEED
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Dataset split summary:")
print(f"  Train : {len(train_df):,}  ({100*len(train_df)/len(data):.1f}%)")
print(f"  Val   : {len(val_df):,}   ({100*len(val_df)/len(data):.1f}%)")
print(f"  Test  : {len(test_df):,}   ({100*len(test_df)/len(data):.1f}%)")

# Confirm class balance is preserved
print("\nSentiment distribution across splits:")
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    dist = split_df["sentiment"].value_counts(normalize=True).sort_index()
    dist_str = ", ".join([f"{SENTIMENT_LABELS[i]}: {v:.2%}" for i, v in dist.items()])
    print(f"  {split_name:6s}: {dist_str}")

---
## Section 4: Preprocessing Pipeline

### 4.1 Text Cleaning

We apply the following cleaning steps:
1. **Lowercase** — reduces vocabulary size
2. **HTML tag removal** — removes `<br/>`, `<p>`, etc.
3. **URL removal** — removes `http://...` patterns
4. **Punctuation normalization** — strip non-alphanumeric characters (keep apostrophes for contractions)
5. **Whitespace normalization** — collapse multiple spaces

In [ ]:
# ── Compiled regex patterns (compile once for efficiency) ─────────────────────
_RE_HTML    = re.compile(r"<[^>]+>")
_RE_URL     = re.compile(r"https?://\S+|www\.\S+")
_RE_NONWORD = re.compile(r"[^a-z0-9'\s]")
_RE_SPACES  = re.compile(r"\s+")
_RE_APOSTROPHE = re.compile(r"'s|n't|'re|'ve|'ll|'d|'m")  # common contractions

def clean_text(text: str) -> str:
    """
    Apply the full text cleaning pipeline:
      1. Lowercase
      2. Remove HTML tags
      3. Remove URLs
      4. Remove non-alphanumeric characters (keep apostrophes)
      5. Normalize whitespace

    Returns the cleaned string (never empty — returns a single space for
    completely empty reviews so downstream code does not crash).
    """
    text = text.lower()
    text = _RE_HTML.sub(" ", text)
    text = _RE_URL.sub(" ", text)
    text = _RE_NONWORD.sub(" ", text)
    text = _RE_SPACES.sub(" ", text).strip()
    return text if text else " "

# Quick test
sample = "This is <b>GREAT</b>! Check http://example.com for deals. Wasn't worth $50."
print("Before:", sample)
print("After :", clean_text(sample))

### 4.2 Tokenization

We use **simple whitespace tokenization** (word-level). After cleaning, the text contains only lowercase alphanumerics and apostrophes separated by spaces, so splitting on whitespace produces clean word tokens.

In [ ]:
def tokenize(text: str) -> list:
    """
    Tokenize a cleaned text string by splitting on whitespace.
    Returns a list of string tokens.
    """
    return text.split()

# Test
sample_clean = clean_text("This is GREAT! Wasn't worth $50.")
print("Tokens:", tokenize(sample_clean))

### 4.3 Vocabulary Construction

Built from **training data only** to prevent leakage. Tokens appearing fewer than `MIN_FREQ` times are excluded and mapped to `<UNK>` at inference time.

In [ ]:
class Vocabulary:
    """
    Word-level vocabulary built from a list of texts.

    Special tokens are always at indices 0-3:
      0: <PAD>  (padding)
      1: <UNK>  (unknown words)
      2: <SOS>  (start of sequence — used by decoder)
      3: <EOS>  (end of sequence   — used by decoder)
    """

    def __init__(self, min_freq: int = 2):
        self.min_freq = min_freq
        self.token2idx: dict = {}
        self.idx2token: dict = {}
        self.token_freq: Counter = Counter()

        # Reserve special token indices
        for i, tok in enumerate(SPECIAL_TOKENS):
            self.token2idx[tok] = i
            self.idx2token[i]   = tok

    @property
    def pad_idx(self): return self.token2idx[PAD_TOKEN]

    @property
    def unk_idx(self): return self.token2idx[UNK_TOKEN]

    @property
    def sos_idx(self): return self.token2idx[SOS_TOKEN]

    @property
    def eos_idx(self): return self.token2idx[EOS_TOKEN]

    def __len__(self):
        return len(self.token2idx)

    def build(self, texts: list):
        """
        Build vocabulary from a list of raw text strings.
        Applies cleaning + tokenization internally.
        Only tokens with freq >= min_freq are added.
        """
        print("Building vocabulary from training data...")
        for text in tqdm(texts, desc="  Counting tokens"):
            tokens = tokenize(clean_text(text))
            self.token_freq.update(tokens)

        next_idx = len(SPECIAL_TOKENS)  # start after reserved special tokens
        added = 0
        for token, freq in sorted(self.token_freq.items()):
            if freq >= self.min_freq and token not in self.token2idx:
                self.token2idx[token] = next_idx
                self.idx2token[next_idx] = token
                next_idx += 1
                added += 1

        print(f"  Unique tokens in training : {len(self.token_freq):,}")
        print(f"  Tokens added (freq>={self.min_freq:d})    : {added:,}")
        print(f"  Total vocabulary size     : {len(self):,}")
        return self

    def numericalize(self, text: str, add_special: bool = False) -> list:
        """
        Convert a raw text string to a list of integer indices.
        Unknown tokens map to unk_idx.
        If add_special=True, prepends SOS and appends EOS.
        """
        tokens = tokenize(clean_text(text))
        indices = [self.token2idx.get(t, self.unk_idx) for t in tokens]
        if add_special:
            indices = [self.sos_idx] + indices + [self.eos_idx]
        return indices

    def decode(self, indices: list) -> str:
        """Convert a list of integer indices back to a token string."""
        return " ".join(self.idx2token.get(i, UNK_TOKEN) for i in indices)

    def save(self, path: Path):
        with open(path, "wb") as f:
            pickle.dump(self, f)
        print(f"Vocabulary saved to {path}")

    @classmethod
    def load(cls, path: Path):
        with open(path, "rb") as f:
            vocab = pickle.load(f)
        print(f"Vocabulary loaded from {path} (size={len(vocab):,})")
        return vocab

In [ ]:
# Build vocabulary from TRAINING data only
vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(train_df["review_text"].tolist())

# Save vocabulary for reuse in later chunks
vocab.save(RESULTS_DIR / "vocabulary.pkl")

In [ ]:
# Sanity checks
assert vocab.pad_idx == 0, "PAD should be index 0"
assert vocab.unk_idx == 1, "UNK should be index 1"
assert vocab.sos_idx == 2, "SOS should be index 2"
assert vocab.eos_idx == 3, "EOS should be index 3"

sample_text = train_df["review_text"].iloc[0]
sample_ids  = vocab.numericalize(sample_text)
print(f"Sample review (first 50 chars): '{sample_text[:50]}...'")
print(f"Numericalized (first 15 ids)  : {sample_ids[:15]}")
print(f"Decoded back (first 15 tokens): {vocab.decode(sample_ids[:15])}")

### 4.4 Padding & Truncation

All sequences are padded (with `<PAD>`) or truncated to exactly `MAX_SEQ_LEN` tokens. Truncation removes tokens from the **end** of the sequence.

In [ ]:
def pad_or_truncate(indices: list, max_len: int, pad_idx: int) -> list:
    """
    Truncate or pad a list of indices to exactly `max_len`.

    - Truncation: removes tokens from the right (end of sequence)
    - Padding:    appends pad_idx tokens on the right

    Returns a list of length exactly max_len.
    """
    if len(indices) >= max_len:
        return indices[:max_len]
    return indices + [pad_idx] * (max_len - len(indices))


def preprocess_text(text: str, vocab: Vocabulary, max_len: int,
                    add_special: bool = False) -> list:
    """
    End-to-end preprocessing for a single review text:
      clean → tokenize → numericalize → pad/truncate

    Args:
        text        : raw review string
        vocab       : built Vocabulary object
        max_len     : fixed output length
        add_special : if True, prepend SOS and append EOS before padding

    Returns:
        A list of integer token indices of length exactly max_len.
    """
    indices = vocab.numericalize(text, add_special=add_special)
    return pad_or_truncate(indices, max_len, vocab.pad_idx)


# Verify
test_ids = preprocess_text(train_df["review_text"].iloc[0], vocab, MAX_SEQ_LEN)
assert len(test_ids) == MAX_SEQ_LEN, f"Expected {MAX_SEQ_LEN}, got {len(test_ids)}"
print(f"Preprocessed sequence length: {len(test_ids)} ✓")
print(f"First 10 indices: {test_ids[:10]}")
print(f"Last  10 indices: {test_ids[-10:]}  (trailing PADs = {test_ids.count(vocab.pad_idx)} total)")

---
## Section 5: Preprocess & Save All Splits

We preprocess the full train, validation, and test sets and save them as NumPy arrays. This avoids re-running preprocessing every time later chunks are executed.

In [ ]:
def preprocess_split(df: pd.DataFrame,
                     vocab: Vocabulary,
                     max_len: int,
                     split_name: str,
                     add_special: bool = False) -> dict:
    """
    Preprocess an entire DataFrame split.

    Returns a dict with:
      'input_ids'    : np.ndarray (N, max_len) of int32 — token indices
      'sentiment'    : np.ndarray (N,)          of int64 — sentiment labels (0/1/2)
      'length_label' : np.ndarray (N,)          of int64 — derived feature labels
      'rating'       : np.ndarray (N,)          of int64 — original star ratings
    """
    print(f"Preprocessing {split_name} split ({len(df):,} samples)...")
    input_ids = np.array(
        [
            preprocess_text(row, vocab, max_len, add_special=add_special)
            for row in tqdm(df["review_text"], desc=f"  {split_name}")
        ],
        dtype=np.int32
    )
    return {
        "input_ids":    input_ids,
        "sentiment":    df["sentiment"].values.astype(np.int64),
        "length_label": df["length_label"].values.astype(np.int64),
        "rating":       df["rating"].values.astype(np.int64),
    }


# Preprocess all splits
train_data = preprocess_split(train_df, vocab, MAX_SEQ_LEN, "Train")
val_data   = preprocess_split(val_df,   vocab, MAX_SEQ_LEN, "Val")
test_data  = preprocess_split(test_df,  vocab, MAX_SEQ_LEN, "Test")

print("\nShapes:")
print(f"  train input_ids : {train_data['input_ids'].shape}")
print(f"  val   input_ids : {val_data['input_ids'].shape}")
print(f"  test  input_ids : {test_data['input_ids'].shape}")

In [ ]:
# ── Save preprocessed arrays ──────────────────────────────────────────────────
for split_name, split_data in [("train", train_data),
                                ("val",   val_data),
                                ("test",  test_data)]:
    for array_name, array in split_data.items():
        path = RESULTS_DIR / f"{split_name}_{array_name}.npy"
        np.save(path, array)

print("All preprocessed arrays saved to results/.")
print("\nSaved files:")
for f in sorted(RESULTS_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:.1f} MB")

---
## Section 6: Preprocessing Analysis & Visualizations

In [ ]:
# ── Review length analysis (before and after truncation) ──────────────────────
raw_lengths   = train_df["review_text"].apply(lambda t: len(tokenize(clean_text(t))))
clipped_lengths = np.minimum(raw_lengths.values, MAX_SEQ_LEN)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(raw_lengths, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(MAX_SEQ_LEN, color="red", linestyle="--", linewidth=1.5,
                label=f"MAX_SEQ_LEN = {MAX_SEQ_LEN}")
axes[0].set_title("Raw Token Length Distribution (Training)", fontweight="bold")
axes[0].set_xlabel("Number of Tokens")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(clipped_lengths, bins=60, color="darkorange", edgecolor="white", alpha=0.85)
axes[1].set_title("Token Length After Truncation", fontweight="bold")
axes[1].set_xlabel("Number of Tokens (capped at MAX_SEQ_LEN)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "token_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

pct_truncated = (raw_lengths > MAX_SEQ_LEN).mean() * 100
pct_padded    = (raw_lengths < MAX_SEQ_LEN).mean() * 100
print(f"Truncated reviews : {pct_truncated:.1f}%")
print(f"Padded reviews    : {pct_padded:.1f}%")
print(f"Mean raw length   : {raw_lengths.mean():.1f} tokens")
print(f"Median raw length : {raw_lengths.median():.1f} tokens")
print(f"Max raw length    : {raw_lengths.max()} tokens")

In [ ]:
# ── Vocabulary coverage on val and test sets ──────────────────────────────────
def compute_coverage(df, vocab):
    """What fraction of tokens in df appear in vocab?"""
    total, known = 0, 0
    for text in df["review_text"]:
        tokens = tokenize(clean_text(text))
        for t in tokens:
            total += 1
            if t in vocab.token2idx:
                known += 1
    return known / total if total else 0

train_cov = compute_coverage(train_df, vocab)
val_cov   = compute_coverage(val_df,   vocab)
test_cov  = compute_coverage(test_df,  vocab)

print("Vocabulary coverage (fraction of tokens in vocab):")
print(f"  Train : {train_cov:.4f}  ({100*train_cov:.2f}%)")
print(f"  Val   : {val_cov:.4f}  ({100*val_cov:.2f}%)")
print(f"  Test  : {test_cov:.4f}  ({100*test_cov:.2f}%)")

In [ ]:
# ── Top 30 most frequent tokens (excluding special tokens) ────────────────────
top_tokens = [
    (tok, freq)
    for tok, freq in vocab.token_freq.most_common()
    if tok not in SPECIAL_TOKENS
][:30]

toks, freqs = zip(*top_tokens)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(toks)), freqs, color="slateblue", edgecolor="white")
ax.set_xticks(range(len(toks)))
ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=9)
ax.set_title("Top 30 Most Frequent Tokens (Training Set)", fontweight="bold")
ax.set_ylabel("Frequency")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(PLOTS_DIR / "top_tokens.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 7: Preprocessing Summary

Print a complete summary of all preprocessing decisions for the report.

In [ ]:
print("=" * 60)
print("PREPROCESSING PIPELINE SUMMARY")
print("=" * 60)
print(f"Dataset categories   : {list(CATEGORIES.keys())}")
print(f"Samples per category : ~{SAMPLES_PER_CATEGORY:,}")
print(f"Total samples        : {len(data):,}")
print()
print("Splits:")
print(f"  Train : {len(train_df):,}  ({100*len(train_df)/len(data):.1f}%)")
print(f"  Val   : {len(val_df):,}    ({100*len(val_df)/len(data):.1f}%)")
print(f"  Test  : {len(test_df):,}    ({100*len(test_df)/len(data):.1f}%)")
print()
print("Cleaning steps:")
print("  1. Lowercase")
print("  2. HTML tag removal")
print("  3. URL removal")
print("  4. Non-alphanumeric character removal (apostrophes kept)")
print("  5. Whitespace normalization")
print()
print("Tokenization : whitespace split on cleaned text")
print(f"Vocabulary   : {len(vocab):,} tokens (min_freq={MIN_FREQ})")
print(f"Special toks : {SPECIAL_TOKENS}")
print(f"Max seq len  : {MAX_SEQ_LEN} tokens")
print(f"Truncated    : {pct_truncated:.1f}% of training reviews")
print(f"Padded       : {pct_padded:.1f}% of training reviews")
print()
print("Tasks:")
print("  Task 1 (Sentiment) : 3-class (Negative/Neutral/Positive)")
print("                       ratings 1-2→0, 3→1, 4-5→2")
print("  Task 2 (Derived)   : Review length category")
print("                       Short(0)=1-30 words, Medium(1)=31-100, Long(2)=101+")
print()
print("Saved artifacts:")
for f in sorted(RESULTS_DIR.glob("*")):
    if f.is_file():
        print(f"  {f}")
print("=" * 60)
print("Chunk 1 complete. Ready for Chunk 2 (Encoder Architecture).")